In [ ]:
#Spark connection
import os
import socket
from pyspark.sql import SparkSession
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = "/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar"
MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        getOrCreate()

In [ ]:
#Иногда бывает нужно добавить в таблицу время совершения операции
from pyspark.sql.functions import current_date, current_timestamp

#Создаем dummy DF
l = [("X", )]
df = spark.createDataFrame(l).toDF("dummy")

df.select(current_date()).show() #yyyy-MM-dd

df.select(current_timestamp()).show(truncate=False) #yyyy-MM-dd HH:mm:ss.SSS

In [ ]:
#Преобразуем string в date и timestamp
from pyspark.sql.functions import lit, to_date, to_timestamp

df.select(to_date(lit('20210228'), 'yyyyMMdd').alias('to_date')).show()

df.select(to_timestamp(lit('20210228 1725'), 'yyyyMMdd HHmm').alias('to_timestamp')).show()

In [ ]:
datetimes = [("2023-02-28", "2023-02-28 10:00:00.123"),
                     ("2023-02-28", "2023-02-28 08:08:08.999"),
                     ("2023-10-31", "2023-12-31 11:59:59.123"),
                     ("2023-11-30", "2023-08-31 00:00:00.000")
                ]

In [ ]:
df_datetimes = spark.createDataFrame(datetimes, schema="date STRING, time STRING")

df_datetimes.show(truncate=False)

In [ ]:
#Прибавляем и вычитаем дни с помощью date_add, date_sub
from pyspark.sql.functions import date_add, date_sub

df_datetimes. \
    withColumn("date_add_date", date_add("date", 10)). \
    withColumn("date_add_time", date_add("time", 10)). \
    withColumn("date_sub_date", date_sub("date", 10)). \
    withColumn("date_sub_time", date_sub("time", 10)). \
    show(truncate=False)

In [ ]:
#Вычисляем разницу во времени
#Используйте help(datediff) для получения доп справки, если необходимо
from pyspark.sql.functions import current_date, current_timestamp, datediff

df_datetimes. \
    withColumn("datediff_date", datediff(current_date(), "date")). \
    withColumn("datediff_time", datediff(current_timestamp(), "time")). \
    show(truncate=False)


In [ ]:
#Доп функции для работы с date

from pyspark.sql.functions import months_between, add_months, round

df_datetimes. \
    withColumn("months_between_date", round(months_between(current_date(), "date"), 2)). \
    withColumn("months_between_time", round(months_between(current_timestamp(), "time"), 2)). \
    withColumn("add_months_date", add_months("date", 3)). \
    withColumn("add_months_time", add_months("time", 3)). \
    show(truncate=False)

In [ ]:
#С помощью trunc можно получить округление даты до первого дня недели, месяца, квартала и года
from pyspark.sql.functions import trunc

df_datetimes. \
    withColumn("date_trunc_week", trunc("date", "week")). \
    withColumn("date_trunc_month", trunc("date", "MM")). \
    withColumn("date_trunc_quarter", trunc("date", "quarter")). \
    withColumn("time_trunc_year", trunc("time", "yy")). \
    show(truncate=False)

help(trunc)

In [ ]:
#Для округления в формате timestamp можно использовать date_trunc
#Принимает параметры 'microsecond', 'millisecond', 'second', 'minute', 'hour', 'week', 'quarter'
from pyspark.sql.functions import date_trunc

df_datetimes. \
    withColumn("date_dt", date_trunc("HOUR", "date")). \
    withColumn("time_dt", date_trunc("HOUR", "time")). \
    withColumn("time_dt1", date_trunc("dd", "time")). \
    show(truncate=False)

help(date_trunc)

In [ ]:
#Решаем задачу извлечения из date различных временных периодов
from pyspark.sql.functions import year, month, weekofyear, dayofmonth, \
    dayofyear, dayofweek, current_date

df_datetimes.select(
    current_date().alias('current_date'), 
    year(current_date()).alias('year'),
    month(current_date()).alias('month'),
    weekofyear(current_date()).alias('weekofyear'),
    dayofyear(current_date()).alias('dayofyear'),
    dayofmonth(current_date()).alias('dayofmonth'),
    dayofweek(current_date()).alias('dayofweek')
).show() #yyyy-MM-dd

In [ ]:
#Решаем задачу извлечения из timestamp различных временных периодов
from pyspark.sql.functions import current_timestamp, hour, minute, second

df_datetimes.select(
    current_timestamp().alias('current_timestamp'), 
    year(current_timestamp()).alias('year'),
    month(current_timestamp()).alias('month'),
    dayofmonth(current_timestamp()).alias('dayofmonth'),
    hour(current_timestamp()).alias('hour'),
    minute(current_timestamp()).alias('minute'),
    second(current_timestamp()).alias('second')
).show(truncate=False) #yyyy-MM-dd HH:mm:ss.SSS

In [ ]:
spark.stop()